# TabPFN 3.0 Learning Curve auf Validierung 2023

Dieses Notebook vergleicht unterschiedliche stage-preserving Trainingsgrößen. Jeder Lauf nutzt `seed=42`, trainiert auf `<=2022` und validiert vollständig auf `2023`. 2024 und 2025 bleiben hier unberührt.


In [1]:
import os
import time
from getpass import getpass
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

DATA_DIR = Path("../../data/processed")
RESULT_DIR = DATA_DIR / "tabpfn_3_experiments"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_PATH = DATA_DIR / "26_cleaned_master_data.pkl"
TARGET = "target_top_10"

FEATURE_COLUMNS = [
    "distance", "vertical_meters", "stage_nr", "team_tier", "age_at_race",
    "rider_bmi", "wind_stability_index", "weather_temp_mean", "weather_temp_trend",
    "weather_rain_prob_mean", "weather_precipitation_mean", "weather_humidity_mean",
    "gradient_final_km", "lag_rider_points_season", "lag_rider_rank_season",
    "lag_race_competitiveness_median", "lag_team_power_index",
]

META_COLUMNS = ["meta_year", "meta_name", "meta_current_team", "meta_race", "stage_nr", "stage_id"]
TARGET_ROWS = [2_000, 4_000, 6_000, 8_000, 10_000, 12_500, 15_000]


In [2]:
def get_best_available_device():
    try:
        import torch

        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"


def clean_tabpfn_token(raw_token):
    token = str(raw_token).strip()
    for prefix in ("export TABPFN_TOKEN=", "TABPFN_TOKEN="):
        if token.startswith(prefix):
            token = token[len(prefix):].strip()
    if (token.startswith('"') and token.endswith('"')) or (token.startswith("'") and token.endswith("'")):
        token = token[1:-1].strip()
    return token


def ensure_tabpfn_ready():
    try:
        import tabpfn
        from tabpfn import TabPFNClassifier
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("TabPFN fehlt. Bitte zuerst ausführen: %pip install -U tabpfn") from exc

    try:
        from packaging.version import Version
        installed = Version(version("tabpfn"))
        if installed < Version("8.0.0"):
            raise RuntimeError(
                f"Installiert ist tabpfn {installed}. Für TabPFN 3 bitte aktualisieren: %pip install -U tabpfn"
            )
    except ImportError:
        print("Hinweis: Paket 'packaging' fehlt; Versionsprüfung wird übersprungen.")

    existing_token = clean_tabpfn_token(os.environ.get("TABPFN_TOKEN", ""))
    if existing_token:
        os.environ["TABPFN_TOKEN"] = existing_token
        print(f"TABPFN_TOKEN ist gesetzt ({len(existing_token)} Zeichen, endet auf ...{existing_token[-4:]}).")
    else:
        raw_token = getpass("TabPFN API Key einfügen; nur den Key, nicht den export-Befehl: ")
        token = clean_tabpfn_token(raw_token)
        if not token:
            raise RuntimeError("TABPFN_TOKEN wurde nicht gesetzt.")
        os.environ["TABPFN_TOKEN"] = token
        print(f"TABPFN_TOKEN wurde gesetzt ({len(token)} Zeichen, endet auf ...{token[-4:]}).")

    os.environ.pop("TABPFN_NO_BROWSER", None)
    print(f"tabpfn-Version: {getattr(tabpfn, '__version__', 'unbekannt')}")
    return TabPFNClassifier


def load_master_data():
    df = pd.read_pickle(MASTER_PATH).copy()
    df["stage_id"] = (
        df["meta_race"].astype(str)
        + "_"
        + df["meta_year"].astype(str)
        + "_ST"
        + df["stage_nr"].astype(str)
    )
    df["target_top_5"] = np.where(df["rank"] <= 5, 1, 0)
    df["target_top_10"] = np.where(df["rank"] <= 10, 1, 0)
    df["target_top_20"] = np.where(df["rank"] <= 20, 1, 0)
    return df


def make_split_bundle(df, mask):
    split = df.loc[mask].copy().reset_index(drop=True)
    return {
        "X": split[FEATURE_COLUMNS].copy(),
        "y": split[TARGET].astype(int).copy(),
        "y_rank": split["rank"].astype(float).copy(),
        "meta": split[META_COLUMNS].copy(),
    }


def load_experiment_splits():
    df = load_master_data()
    return {
        "train_selection": make_split_bundle(df, df["meta_year"] <= 2022),
        "validation": make_split_bundle(df, df["meta_year"] == 2023),
        "test_original": make_split_bundle(df, df["meta_year"].isin([2024, 2025])),
        "train_final": make_split_bundle(df, df["meta_year"] <= 2023),
    }


def summarize_split(name, bundle):
    meta = bundle["meta"]
    y = bundle["y"]
    stage_counts = meta.groupby("stage_id").size()
    return {
        "split": name,
        "rows": len(meta),
        "years": ", ".join(map(str, sorted(meta["meta_year"].unique()))),
        "n_stages": meta["stage_id"].nunique(),
        "stage_rows_min": int(stage_counts.min()),
        "stage_rows_median": float(stage_counts.median()),
        "stage_rows_mean": float(stage_counts.mean()),
        "stage_rows_max": int(stage_counts.max()),
        "target_positive_rate": float(y.mean()),
    }


def make_stage_preserving_subset(X, y, meta, target_rows, seed):
    stage_counts = meta.groupby("stage_id", sort=False).size().reset_index(name="rows")
    rng = np.random.default_rng(seed)
    shuffled_stage_ids = stage_counts["stage_id"].to_numpy(copy=True)
    rng.shuffle(shuffled_stage_ids)

    selected_stage_ids = []
    selected_rows = 0
    row_lookup = stage_counts.set_index("stage_id")["rows"].to_dict()

    for stage_id in shuffled_stage_ids:
        selected_stage_ids.append(stage_id)
        selected_rows += int(row_lookup[stage_id])
        if selected_rows >= target_rows:
            break

    selected_stage_ids = set(selected_stage_ids)
    mask = meta["stage_id"].isin(selected_stage_ids)

    X_sub = X.loc[mask].reset_index(drop=True)
    y_sub = y.loc[mask].reset_index(drop=True)
    meta_sub = meta.loc[mask].reset_index(drop=True)

    validate_stage_preserving_sample(meta, meta_sub, target_rows)

    sample_info = {
        "target_rows": int(target_rows),
        "actual_rows": int(len(meta_sub)),
        "row_overshoot": int(len(meta_sub) - target_rows),
        "n_stages": int(meta_sub["stage_id"].nunique()),
        "seed": int(seed),
        "positive_rate": float(y_sub.mean()),
    }
    return X_sub, y_sub, meta_sub, sample_info


def validate_stage_preserving_sample(source_meta, sample_meta, target_rows):
    if len(sample_meta) < target_rows:
        raise AssertionError(f"Sample hat nur {len(sample_meta)} Zeilen, Ziel war mindestens {target_rows}.")

    source_counts = source_meta.groupby("stage_id").size()
    sample_counts = sample_meta.groupby("stage_id").size()

    mismatched = []
    for stage_id, sample_count in sample_counts.items():
        source_count = int(source_counts.loc[stage_id])
        if int(sample_count) != source_count:
            mismatched.append((stage_id, int(sample_count), source_count))

    if mismatched:
        raise AssertionError(f"Teilweise Stage-Auswahl gefunden: {mismatched[:5]}")
    return True


def predict_positive_proba(model, X, batch_size):
    proba_parts = []
    current_batch_size = batch_size
    start_idx = 0

    while start_idx < len(X):
        stop_idx = min(start_idx + current_batch_size, len(X))
        X_batch = X.iloc[start_idx:stop_idx]
        try:
            proba_parts.append(model.predict_proba(X_batch)[:, 1])
            print(f"Vorhergesagt: {stop_idx:,}/{len(X):,} Zeilen (Batch: {current_batch_size})")
            start_idx = stop_idx
        except Exception as exc:
            name = type(exc).__name__.lower()
            message = str(exc).lower()
            is_oom = "outofmemory" in name or "out of memory" in message
            if not is_oom or current_batch_size <= 10:
                raise
            current_batch_size = max(10, current_batch_size // 2)
            print(f"Speichergrenze erreicht. Neuer Prediction-Batch: {current_batch_size}")

    return np.concatenate(proba_parts)


def build_prediction_frame(meta, y_rank, y_true, proba, proba_col="p_top10_tabpfn"):
    pred = meta.reset_index(drop=True).copy()
    pred["actual_rank"] = y_rank.reset_index(drop=True).astype(float)
    pred["actual_top10"] = y_true.reset_index(drop=True).astype(int)
    pred[proba_col] = proba
    pred["pred_top10_0_50"] = (pred[proba_col] >= 0.50).astype(int)
    pred["rank_within_stage_by_model"] = (
        pred.groupby("stage_id")[proba_col]
        .rank(ascending=False, method="first")
        .astype(int)
    )
    return pred


def get_stage_top10_predictions(pred, proba_col="p_top10_tabpfn"):
    return (
        pred.sort_values(["stage_id", proba_col], ascending=[True, False])
        .groupby("stage_id", as_index=False)
        .head(10)
        .reset_index(drop=True)
    )


def evaluate_prediction_frame(pred, proba_col="p_top10_tabpfn"):
    y_true = pred["actual_top10"].astype(int)
    proba = pred[proba_col].astype(float)
    hard = (proba >= 0.50).astype(int)

    stage_top10 = get_stage_top10_predictions(pred, proba_col)
    actual_top10_per_stage = pred.groupby("stage_id")["actual_top10"].sum().rename("actual_top10_count")
    stage_eval = (
        stage_top10.groupby("stage_id")
        .agg(
            actual_top10_in_predicted_top10=("actual_top10", "sum"),
            winner_in_predicted_top10=("actual_rank", lambda s: bool((s == 1).any())),
        )
        .join(actual_top10_per_stage)
        .reset_index()
    )
    stage_eval["stage_top10_recall"] = (
        stage_eval["actual_top10_in_predicted_top10"] / stage_eval["actual_top10_count"].replace(0, np.nan)
    )

    return {
        "rows": int(len(pred)),
        "n_stages": int(pred["stage_id"].nunique()),
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "average_precision": float(average_precision_score(y_true, proba)),
        "precision_at_0_50": float(precision_score(y_true, hard, zero_division=0)),
        "recall_at_0_50": float(recall_score(y_true, hard, zero_division=0)),
        "mean_stage_top10_recall": float(stage_eval["stage_top10_recall"].mean()),
        "winner_hit_rate_in_predicted_top10": float(stage_eval["winner_in_predicted_top10"].mean()),
    }


def evaluate_by_year(pred, proba_col="p_top10_tabpfn"):
    rows = []
    for year, pred_year in pred.groupby("meta_year"):
        metrics = evaluate_prediction_frame(pred_year.copy(), proba_col)
        metrics["year"] = int(year)
        rows.append(metrics)
    return pd.DataFrame(rows).sort_values("year")


## Konfiguration

Alle Zielgrößen werden als Mindestwerte interpretiert. Durch vollständige `stage_id`s kann die tatsächliche Zeilenzahl etwas höher sein.


In [3]:
DEVICE = get_best_available_device()
TabPFNClassifier = ensure_tabpfn_ready()

TRAIN_TARGET_ROWS = [2_000, 4_000, 6_000, 8_000, 10_000, 12_500, 15_000]
SEED = 42
PREDICTION_BATCH_SIZE = 100
FORCE_RERUN = False

print(f"Device: {DEVICE}")
print(f"Train target rows: {TRAIN_TARGET_ROWS}")
print(f"Prediction batch size: {PREDICTION_BATCH_SIZE}")


TABPFN_TOKEN wurde gesetzt (53 Zeichen, endet auf ...Kr7o).
tabpfn-Version: 8.0.8
Device: mps
Train target rows: [2000, 4000, 6000, 8000, 10000, 12500, 15000]
Prediction batch size: 100


In [4]:
splits = load_experiment_splits()
train_bundle = splits["train_selection"]
val_bundle = splits["validation"]

print(f"Training pool <=2022: {train_bundle['X'].shape}")
print(f"Validation 2023     : {val_bundle['X'].shape}")
print(f"Positive rate train : {train_bundle['y'].mean():.2%}")
print(f"Positive rate val   : {val_bundle['y'].mean():.2%}")


Training pool <=2022: (169349, 17)
Validation 2023     : (8897, 17)
Positive rate train : 5.91%
Positive rate val   : 6.41%


## Learning-Curve-Läufe

Pro Zielgröße wird ein eigener stage-preserving Trainingskontext erzeugt. Jede Ergebnisdatei enthält die tatsächliche Zeilenzahl und Anzahl gezogener Stages.


In [5]:
all_metrics = []

for target_rows in TRAIN_TARGET_ROWS:
    run_label = f"lc_val2023_minrows_{target_rows}_seed_{SEED}"
    pred_path = RESULT_DIR / f"{run_label}_predictions.csv"
    stage_pred_path = RESULT_DIR / f"{run_label}_stage_top10_predictions.csv"
    metrics_path = RESULT_DIR / f"{run_label}_metrics.csv"

    print("=" * 80)
    print(f"Run: {run_label}")
    start_total = time.time()

    if pred_path.exists() and not FORCE_RERUN:
        print(f"Lade vorhandene Predictions: {pred_path}")
        pred = pd.read_csv(pred_path)
        sample_info = {
            "target_rows": target_rows,
            "actual_rows": np.nan,
            "row_overshoot": np.nan,
            "n_stages": np.nan,
            "seed": SEED,
            "positive_rate": np.nan,
        }
        fit_seconds = np.nan
        predict_seconds = np.nan
        source = "cached"
    else:
        X_fit, y_fit, meta_fit, sample_info = make_stage_preserving_subset(
            train_bundle["X"],
            train_bundle["y"],
            train_bundle["meta"],
            target_rows=target_rows,
            seed=SEED,
        )
        print(
            f"Fit-Set: {sample_info['actual_rows']:,} Zeilen "
            f"aus {sample_info['n_stages']} Stages; Top-10-Anteil {sample_info['positive_rate']:.2%}"
        )

        clf = TabPFNClassifier(device=DEVICE)
        start_fit = time.time()
        clf.fit(X_fit, y_fit)
        fit_seconds = time.time() - start_fit

        start_pred = time.time()
        proba = predict_positive_proba(clf, val_bundle["X"], PREDICTION_BATCH_SIZE)
        predict_seconds = time.time() - start_pred

        pred = build_prediction_frame(val_bundle["meta"], val_bundle["y_rank"], val_bundle["y"], proba)
        pred.to_csv(pred_path, index=False)
        get_stage_top10_predictions(pred).to_csv(stage_pred_path, index=False)
        source = "computed"

    metrics = evaluate_prediction_frame(pred)
    metrics.update({
        "run_label": run_label,
        "phase": "validation_2023_learning_curve",
        "target": TARGET,
        "device": DEVICE,
        "target_rows": target_rows,
        "actual_train_rows": sample_info["actual_rows"],
        "row_overshoot": sample_info["row_overshoot"],
        "train_stages": sample_info["n_stages"],
        "seed": SEED,
        "prediction_batch_size": PREDICTION_BATCH_SIZE,
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        "total_seconds_notebook_step": time.time() - start_total,
        "source": source,
    })
    pd.DataFrame([metrics]).to_csv(metrics_path, index=False)
    all_metrics.append(metrics)

learning_curve_metrics = pd.DataFrame(all_metrics).sort_values("target_rows")
learning_curve_metrics["delta_stage_top10_recall"] = learning_curve_metrics["mean_stage_top10_recall"].diff()
learning_curve_metrics["delta_average_precision"] = learning_curve_metrics["average_precision"].diff()
learning_curve_metrics.to_csv(RESULT_DIR / "learning_curve_val2023_summary.csv", index=False)
display(learning_curve_metrics)


Run: lc_val2023_minrows_2000_seed_42
Fit-Set: 2,041 Zeilen aus 12 Stages; Top-10-Anteil 5.98%
Vorhergesagt: 100/8,897 Zeilen (Batch: 100)
Vorhergesagt: 200/8,897 Zeilen (Batch: 100)
Vorhergesagt: 300/8,897 Zeilen (Batch: 100)
Vorhergesagt: 400/8,897 Zeilen (Batch: 100)
Vorhergesagt: 500/8,897 Zeilen (Batch: 100)
Vorhergesagt: 600/8,897 Zeilen (Batch: 100)
Vorhergesagt: 700/8,897 Zeilen (Batch: 100)
Vorhergesagt: 800/8,897 Zeilen (Batch: 100)
Vorhergesagt: 900/8,897 Zeilen (Batch: 100)
Vorhergesagt: 1,000/8,897 Zeilen (Batch: 100)
Vorhergesagt: 1,100/8,897 Zeilen (Batch: 100)
Vorhergesagt: 1,200/8,897 Zeilen (Batch: 100)
Vorhergesagt: 1,300/8,897 Zeilen (Batch: 100)
Vorhergesagt: 1,400/8,897 Zeilen (Batch: 100)
Vorhergesagt: 1,500/8,897 Zeilen (Batch: 100)
Vorhergesagt: 1,600/8,897 Zeilen (Batch: 100)
Vorhergesagt: 1,700/8,897 Zeilen (Batch: 100)
Vorhergesagt: 1,800/8,897 Zeilen (Batch: 100)
Vorhergesagt: 1,900/8,897 Zeilen (Batch: 100)
Vorhergesagt: 2,000/8,897 Zeilen (Batch: 100)
Vorh

,rows,n_stages,roc_auc,average_precision,precision_at_0_50,recall_at_0_50,mean_stage_top10_recall,winner_hit_rate_in_predicted_top10,run_label,phase,...,row_overshoot,train_stages,seed,prediction_batch_size,fit_seconds,predict_seconds,total_seconds_notebook_step,source,delta_stage_top10_recall,delta_average_precision
0,8897,57,0.732774,0.209647,0.474359,0.064912,0.291813,0.421053,lc_val2023_minrows_2000_seed_42,validation_2023_learning_curve,...,41,12,42,100,0.487982,590.309830,591.061851,computed,NaN,NaN
1,8897,57,0.736901,0.226244,0.547170,0.050877,0.302144,0.456140,lc_val2023_minrows_4000_seed_42,validation_2023_learning_curve,...,86,24,42,100,0.554775,1173.300636,1173.916135,computed,0.010331,0.016597
2,8897,57,0.752517,0.247950,0.512821,0.070175,0.314620,0.456140,lc_val2023_minrows_6000_seed_42,validation_2023_learning_curve,...,93,36,42,100,0.465320,2462.155212,2462.686794,computed,0.012476,0.021706
3,8897,57,0.759255,0.245561,0.523810,0.077193,0.316374,0.456140,lc_val2023_minrows_8000_seed_42,validation_2023_learning_curve,...,118,48,42,100,0.496154,4736.549455,4737.113710,computed,0.001754,-0.002389
4,8897,57,0.766528,0.255855,0.521127,0.064912,0.314620,0.456140,lc_val2023_minrows_10000_seed_42,validation_2023_learning_curve,...,100,60,42,100,0.493696,3397.077263,3397.640643,computed,-0.001754,0.010295
5,8897,57,0.767758,0.254961,0.513889,0.064912,0.321637,0.473684,lc_val2023_minrows_12500_seed_42,validation_2023_learning_curve,...,73,75,42,100,0.483368,4570.399993,4570.958221,computed,0.007018,-0.000894
6,8897,57,0.773636,0.261439,0.602941,0.071930,0.318129,0.456140,lc_val2023_minrows_15000_seed_42,validation_2023_learning_curve,...,9,90,42,100,0.494887,5928.326485,5928.908467,computed,-0.003509,0.006478


## Auswahlhilfe

Die Tabelle sortiert Performance und Laufzeit zusammen. Die finale Entscheidung für `SELECTED_TARGET_ROWS` wird anschließend in `10-02_03` eingetragen.


In [6]:
summary = pd.read_csv(RESULT_DIR / "learning_curve_val2023_summary.csv")
cols = [
    "target_rows", "actual_train_rows", "train_stages", "row_overshoot",
    "roc_auc", "average_precision", "mean_stage_top10_recall",
    "winner_hit_rate_in_predicted_top10", "predict_seconds", "source",
]
display(summary[cols].sort_values("target_rows"))

best_stage_recall = summary.sort_values(
    ["mean_stage_top10_recall", "average_precision"],
    ascending=False,
).iloc[0]
print(
    "Beste Validierungsgröße nach stage_top10_recall: "
    f"{int(best_stage_recall['target_rows']):,} Mindestzeilen "
    f"({int(best_stage_recall['actual_train_rows']) if pd.notna(best_stage_recall['actual_train_rows']) else 'cached'} tatsächlich)."
)


,target_rows,actual_train_rows,train_stages,row_overshoot,roc_auc,average_precision,mean_stage_top10_recall,winner_hit_rate_in_predicted_top10,predict_seconds,source
0,2000,2041,12,41,0.732774,0.209647,0.291813,0.421053,590.309830,computed
1,4000,4086,24,86,0.736901,0.226244,0.302144,0.456140,1173.300636,computed
2,6000,6093,36,93,0.752517,0.247950,0.314620,0.456140,2462.155212,computed
3,8000,8118,48,118,0.759255,0.245561,0.316374,0.456140,4736.549455,computed
4,10000,10100,60,100,0.766528,0.255855,0.314620,0.456140,3397.077263,computed
5,12500,12573,75,73,0.767758,0.254961,0.321637,0.473684,4570.399993,computed
6,15000,15009,90,9,0.773636,0.261439,0.318129,0.456140,5928.326485,computed


Beste Validierungsgröße nach stage_top10_recall: 12,500 Mindestzeilen (12573 tatsächlich).
